# Environment Setup

## Kernel Installation

In [1]:
%%bash
# Install Jupyter kernel in the virtual environment
source .env/bin/activate 
uv pip install ipykernel -q

# Install custom kernel
python -m ipykernel install --user --name=mlops --display-name="Python (mlops)"

Installed kernelspec mlops in /home/jupyter/.local/share/jupyter/kernels/mlops


## Configure Environment Variables

In [1]:
import os
import warnings

# Setup environment path for consistent package management
os.environ['PATH'] = os.path.abspath('.env/bin') + ':' + os.environ.get('PATH', '')

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")
%env PYTHONWARNINGS=ignore
%env JUPYTER_PLATFORM_DIRS=1

env: PYTHONWARNINGS=ignore
env: JUPYTER_PLATFORM_DIRS=1


## Install Required Dependencies

In [2]:
%%bash
# Install Feast with GCP support and scikit-learn
# -q flag for quiet installation
uv pip install feast['gcp'] scikit-learn -q

## GCP Configuration

In [3]:
# Google Cloud Platform project configuration for this assignment
PROJECT_ID = "arcane-rigging-461217-m1"  # GCP project ID
TABLE_ID = "Iris_Dataset.Iris_Table"     # BigQuery source table containing the iris dataset
BUCKET_ID = "mlops-course-arcane-rigging-461217-m1"  # GCS bucket for registry
BIGQUERY_DATASET_NAME = "Mlops_offline_store"  # BigQuery dataset name for offline store
DATASTORE_NAMESPACE = "Mlops_online_store"     # Cloud Datastore namespace for online store

# Prepare Datasets in the Required Format

## Add Timestamp column to the Data

In [4]:
# Import required libraries
import pandas as pd
from datetime import datetime, timedelta

# Load the Iris dataset
data = pd.read_csv('data/iris.csv')
print(f"Original dataset shape: {data.shape}")

# Add timestamps for point-in-time feature serving
# Create timestamps spaced 5 minutes apart for each record
start_date = datetime.now()
timestamps = [start_date + timedelta(minutes=i*5) for i in range(len(data))]
data['event_timestamp'] = timestamps

print("Dataset with timestamps:")
data.head(10)

Original dataset shape: (150, 5)
Dataset with timestamps:


,sepal_length,sepal_width,petal_length,petal_width,species,event_timestamp
0,5.1,3.5,1.4,0.2,setosa,2025-06-21 19:22:06.043098
1,4.9,3.0,1.4,0.2,setosa,2025-06-21 19:27:06.043098
2,4.7,3.2,1.3,0.2,setosa,2025-06-21 19:32:06.043098
3,4.6,3.1,1.5,0.2,setosa,2025-06-21 19:37:06.043098
4,5.0,3.6,1.4,0.2,setosa,2025-06-21 19:42:06.043098
5,5.4,3.9,1.7,0.4,setosa,2025-06-21 19:47:06.043098
6,4.6,3.4,1.4,0.3,setosa,2025-06-21 19:52:06.043098
7,5.0,3.4,1.5,0.2,setosa,2025-06-21 19:57:06.043098
8,4.4,2.9,1.4,0.2,setosa,2025-06-21 20:02:06.043098
9,4.9,3.1,1.5,0.1,setosa,2025-06-21 20:07:06.043098


## Create Entity DataFrame

In [5]:
# Create entity dataframe for Feast
# Remove the last record from each species group to ensure 
# the latest record of each entity is not used for model training
# so that online inferencing can be done
filtered_data = data.groupby('species').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)

# Create entity dataframe with species and timestamps
# This will be used for getting historical features
result = filtered_data[['species', 'event_timestamp']]

# Save entity dataframe for later use in model training
result.to_csv("data/entity.csv", index=False)
print(f"Entity dataframe created with {len(result)} records")
print("Entity dataframe preview:")
result.head()

Entity dataframe created with 147 records
Entity dataframe preview:


,species,event_timestamp
0,setosa,2025-06-21 19:22:06.043098
1,setosa,2025-06-21 19:27:06.043098
2,setosa,2025-06-21 19:32:06.043098
3,setosa,2025-06-21 19:37:06.043098
4,setosa,2025-06-21 19:42:06.043098


## Upload Data to BigQuery Source Table

In [6]:
# Creating the BigQuery source table
import pandas_gbq

# Define table schema for BigQuery
table_schema = [
    {'name': 'sepal_length', 'type': 'FLOAT'},
    {'name': 'sepal_width', 'type': 'FLOAT'}, 
    {'name': 'petal_length', 'type': 'FLOAT'},
    {'name': 'petal_width', 'type': 'FLOAT'},
    {'name': 'species', 'type': 'STRING'},
    {'name': 'event_timestamp', 'type': 'TIMESTAMP'}
]

# Upload dataframe to BigQuery
# if_exists="replace" will overwrite existing table
pandas_gbq.to_gbq(
    data, 
    TABLE_ID, 
    project_id=PROJECT_ID, 
    if_exists="replace", 
    table_schema=table_schema
)

print(f"Data successfully uploaded to BigQuery table: {TABLE_ID}")
print(f"Table contains {len(data)} records")

100%|██████████| 1/1 [00:00<00:00, 8738.13it/s]

Data successfully uploaded to BigQuery table: Iris_Dataset.Iris_Table
Table contains 150 records


# Feast Related Setup

## Initialize Feast Repository

In [7]:
%%bash
# Initialize the Feast repository with GCP template
feast init -m Iris_Feast -t gcp


Creating a new Feast repository in /home/jupyter/assignment/Iris_Feast.



In [8]:
# Change to feature repository directory
%cd Iris_Feast/feature_repo

/home/jupyter/assignment/Iris_Feast/feature_repo


## Configure Feature Store

In [9]:
# Create feature store configuration
# This configures Feast to use BigQuery as offline store and Datastore as online store
feature_store = f"""project: Iris_Feast
registry: gs://{BUCKET_ID}/feast/registry.db
provider: gcp
entity_key_serialization_version: 2

offline_store:
  type: bigquery
  dataset: {BIGQUERY_DATASET_NAME}

online_store:
  type: datastore
  project_id: {PROJECT_ID}
  namespace: {DATASTORE_NAMESPACE}
"""

# Write configuration to feature_store.yaml
with open('feature_store.yaml', "w") as feature_store_file:
    feature_store_file.write(feature_store)
    
print("Feature store configuration created successfully!")

Feature store configuration created successfully!


## Define Feast Objects

In [15]:
# Creates definitions of entity, feature view, and feature service
flower_features = f"""
from datetime import timedelta
from feast import BigQuerySource, FeatureView, FeatureService, Entity, ValueType

# Define flower species as entity
flower_entity = Entity(
    name="species",
    description="A Species of Iris Flower",
    value_type=ValueType.STRING
)

# Define feature view for flower measurements
flower_features = FeatureView(
    name="flower_features",
    entities=[flower_entity],
    ttl=timedelta(weeks=52),  # Time-to-live for features
    source=BigQuerySource(
        table=f"{PROJECT_ID}.{TABLE_ID}",
        timestamp_field="event_timestamp"
    ),
    tags={{"assignment":"week_3"}}
)

# Create feature service for one model version
# FeatureService groups features for specific use cases
model_v1 = FeatureService(
    name="feast_model_v1",
    features=[flower_features]
)
"""

# Write feature definitions to feature_repo.py
with open('feature_repo.py', "w") as feature_repo_file:
    feature_repo_file.write(flower_features)
    
print("Feature repository definitions created successfully!")

Feature repository definitions created successfully!


## Deploy Feature Store

In [24]:
# Sync the metadata about Feast objects to the registry
# and Create all necessary feature store infrastructure
!feast apply

/home/jupyter/assignment/.env/lib/python3.12/site-packages/feast/repo_config.py:268: DeprecationWarning: The serialization version 2 and below will be deprecated in the next release. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
No project found in the repository. Using project name Iris_Feast defined in feature_store.yaml
Applying changes for project Iris_Feast
Deploying infrastructure for flower_features


## Materialize Features to Online Store

In [25]:
# Load latest data from the feature view into the online store between these two dates
!feast materialize 2025-06-20 2025-06-23

/home/jupyter/assignment/.env/lib/python3.12/site-packages/feast/repo_config.py:268: DeprecationWarning: The serialization version 2 and below will be deprecated in the next release. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
Materializing 1 feature views from 2025-06-20 00:00:00+00:00 to 2025-06-23 00:00:00+00:00 into the datastore online store.

flower_features:
100%|█████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  8.89it/s]
